# SER · Session 1 — Dataset validation and feature extraction

Run this on a **CPU** session. CPU sessions do **not** consume GPU quota, and
feature extraction is pure CPU work — running it on a GPU session would burn
2–3 GPU-hours while the GPU sits idle.

**Attach these four datasets** (Add Input → Datasets):

| Corpus | Slug |
|---|---|
| RAVDESS | `uwrfkaggler/ravdess-emotional-speech-audio` |
| TESS | `ejlok1/toronto-emotional-speech-set-tess` |
| SAVEE | `ejlok1/surrey-audiovisual-expressed-emotion-savee` |
| CREMA-D | `ejlok1/cremad` |

When it finishes, publish `/kaggle/working/features_cache` as a new Kaggle
Dataset and attach that to notebook 02.

In [ ]:
import glob
import json
import os
import shutil
import sys

import tensorflow as tf
import keras

print("python", sys.version.split()[0])
print("TF    ", tf.__version__)
print("Keras ", keras.__version__)
print("GPU   ", tf.config.list_physical_devices("GPU") or "none (expected)")
print()
print("Keras 3 in use:", keras.__version__.startswith("3"))

In [ ]:
REPO = "https://github.com/Eldorado5002/ser.git"

if not os.path.exists("/kaggle/working/ser"):
    !git clone -q {REPO} /kaggle/working/ser

sys.path.insert(0, "/kaggle/working/ser")
os.chdir("/kaggle/working/ser")
!git log --oneline -1

In [ ]:
# PINNED path. Both notebooks MUST use this identical string: the feature
# cache fingerprint hashes ABSOLUTE file paths, so a different root silently
# invalidates every cache and re-extracts features on GPU time.
DATA_ROOT = "/kaggle/working/ser/data"

SOURCES = {
    "RAVDESS": "/kaggle/input/ravdess-emotional-speech-audio/audio_speech_actors_01-24",
    "TESS":    "/kaggle/input/toronto-emotional-speech-set-tess/TESS Toronto emotional speech set data",
    "SAVEE":   "/kaggle/input/surrey-audiovisual-expressed-emotion-savee/ALL",
    "CREMA-D": "/kaggle/input/cremad/AudioWAV",
}

os.makedirs(DATA_ROOT, exist_ok=True)
for name, src in SOURCES.items():
    dst = os.path.join(DATA_ROOT, name)
    assert os.path.isdir(src), (
        f"MISSING INPUT: {src}\n"
        f"Attach the dataset, then check the mount path under /kaggle/input/.")
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        shutil.rmtree(dst)
    os.symlink(src, dst)
    print(f"{name:9s} -> {src}")

In [ ]:
import config
from data_loader import EXPECTED_COUNTS, build_metadata, split_metadata

# Hard gate. RAVDESS and TESS mirrors each ship the corpus TWICE; scanning a
# duplicated tree would leak identical audio across the train/test split.
for name in SOURCES:
    n = len(glob.glob(os.path.join(DATA_ROOT, name, "**", "*.wav"),
                      recursive=True))
    exp = EXPECTED_COUNTS[name]
    print(f"{name:9s} {n:6d} wav (expected {exp:6d})  "
          f"{'OK' if n == exp else 'MISMATCH'}")
    assert n == exp, (
        f"{name}: found {n} wav files, expected {exp}. The symlink probably "
        f"points at a duplicated tree - see data_loader.CANONICAL_SUBDIRS.")

meta = build_metadata(strict=True)
assert len(meta) == 12162, f"expected 12162 samples, got {len(meta)}"
assert config.INPUT_LEN == 2376, f"input length is {config.INPUT_LEN}"

print("\nVALIDATION PASSED - 12,162 samples, input length 2376")
print(meta.groupby("emotion").size().to_string())

In [ ]:
import numpy as np
from model import AdaptiveFeatureWeighting, build_model

# Kaggle ships its own TensorFlow, which may be Keras 3 while the local test
# suite pins Keras 2. Verify the custom AFW layer survives a save/load round
# trip HERE, in a free CPU session, before committing any GPU hours.
m, wm = build_model(use_afw=True, use_mstc=True)
x = [np.random.randn(2, config.MFCC_LEN).astype("float32"),
     np.random.randn(2, config.ZCR_LEN).astype("float32"),
     np.random.randn(2, config.RMSE_LEN).astype("float32")]
before = m.predict(x, verbose=0)

m.save("/tmp/roundtrip.keras")
r = tf.keras.models.load_model(
    "/tmp/roundtrip.keras",
    custom_objects={"AdaptiveFeatureWeighting": AdaptiveFeatureWeighting},
    compile=False)
np.testing.assert_allclose(before, r.predict(x, verbose=0), rtol=1e-5)

print(f"Custom layer survives save/load on Keras {keras.__version__}")
print(f"{m.count_params():,} parameters")

In [ ]:
from augmentation import plan_augmentation
from features import build_feature_matrix, df_to_items

config.CACHE_DIR = "/kaggle/working/features_cache"
os.makedirs(config.CACHE_DIR, exist_ok=True)

train_df, val_df, test_df = split_metadata(meta)

# BOTH augmentation policies are required: the ablation runs EAAA configs
# (eaaa, full) and uniform configs (base, afw, mstc, cadl).
for aware, key in ((True, "eaaa"), (False, "uniform")):
    items = plan_augmentation(train_df, emotion_aware=aware)
    build_feature_matrix(items, desc=f"train_{key}")

build_feature_matrix(df_to_items(val_df), desc="val")
build_feature_matrix(df_to_items(test_df), desc="test")

print()
for f in sorted(glob.glob(f"{config.CACHE_DIR}/*.npz")):
    print(f"{os.path.basename(f):52s} {os.path.getsize(f)/1e6:7.1f} MB")

In [ ]:
manifest = {
    "data_root": DATA_ROOT,
    "n_samples": int(len(meta)),
    "input_len": int(config.INPUT_LEN),
    "splits": {"train": int(len(train_df)), "val": int(len(val_df)),
               "test": int(len(test_df))},
    "caches": [os.path.basename(f)
               for f in sorted(glob.glob(f"{config.CACHE_DIR}/*.npz"))],
    "tf_version": tf.__version__,
    "keras_version": keras.__version__,
}
with open(f"{config.CACHE_DIR}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))

## Next steps

1. **Save Version → Save & Run All (Commit)** — this runs headless to
   completion, so you can close the browser. An interactive session gets
   reclaimed after ~20–40 minutes idle.
2. When it finishes, publish `/kaggle/working/features_cache` as a **new
   Kaggle Dataset** (e.g. `ser-feature-cache`).
3. Attach that dataset to `02_train.ipynb`, set **Accelerator → GPU**, and run.